# Lab Seven: Spatial Microsimulation - Simple World

In [ ]:
import pandas as pd
import numpy as np

# Define number of individuals
num_individuals = 1000

# Generate synthetic data using a random function.
ages = np.random.randint(2, 90, size=num_individuals)
sex = np.random.choice(['m', 'f'], size=num_individuals)
income = np.random.randint(1000, 10000, size=num_individuals)

# Create DataFrame
synthetic_data = pd.DataFrame({'age': ages, 'sex': sex, 'income': income})
synthetic_data.head()

In [ ]:
#!pip install ipfn
# run this once - then replace the hash key.

In [ ]:
# Load the individual level data
ind = pd.read_csv('data/SimpleWorld/ind-full.csv')
ind

In [ ]:
con_sex = pd.read_csv('data/SimpleWorld/sex.csv')
con_age = pd.read_csv('data/SimpleWorld/age.csv')

In [ ]:
con_age

In [ ]:
con_sex

In [ ]:
# Check that the totals of the two constraints tables match
print ("con_age total:", con_age.values.sum(), "; con_sex total:", con_sex.values.sum())

# Check the row totals (i.e. area populations match)
print ("con_age row sums:", con_age.sum(axis=1).values, "; con_sex row sums:", con_sex.sum(axis=1).values)

# Test row sum equivalence
con_age.sum(axis=1) == con_sex.sum(axis=1)

In [ ]:
# Store the full individidual dataset as ind_orig. 
# Use copy do make a 'deep' copy.
ind_orig = ind.copy()
# Drop the income field
ind = ind.drop(['income'],axis=1)
ind

In [ ]:
# Now recategorise the age variable
ind['age'] = pd.to_numeric(ind['age'])

# overwrite the age variable with categorical age
ind['age'] = pd.cut(ind['age'], [0,49,120], labels = ['a0_49','a50+'])
ind

In [ ]:
ind.age

In [ ]:
con_age.columns

In [ ]:
# Rename the con_age fields to match the categories in the ind table
con_age = con_age.rename(columns={'a0.49':'a0_49','a.50+':'a50+'})
con_age

In [ ]:
# Finally create a single constraint object by merging the constraints tables.
cons = con_age.merge(con_sex,left_index=True,right_index=True)
cons

In [ ]:
# Check the dimensions of the ind and cons datasets
print ("Shape of ind:",ind.shape)
print ("Shape of cons:",cons.shape)

In [ ]:
# we need to 'flatten' the individual dataset as the dimensions differ.
# This means that responses become fields, and values become booleans, with rows reflecting individuals
age_pivot = pd.pivot_table(ind,columns=['age'],values='id', index=ind.index, aggfunc=len, fill_value=0, observed=False )

# The last square bracket bit ensures that the column order is male then female.
sex_pivot = pd.pivot_table(ind,columns=['sex'],values='id', index=ind.index, aggfunc=len, fill_value=0, observed=False )[['m','f']]

In [ ]:
age_pivot

In [ ]:
# merge pivoted data to make flatten dataframe
ind_cat = pd.DataFrame(age_pivot.to_records()).merge(pd.DataFrame(sex_pivot.to_records()),left_index=True,right_index=True)
# drop intermediate columns
ind_cat = ind_cat.drop(['index_x','index_y'],axis=1)
ind_cat

In [ ]:
# Check the columns sums to be sure ind_cat is correct
ind_cat.sum(axis=0)

# store these values
ind_agg = ind_cat.sum(axis=0)
ind_agg

# Now the survey data is in the same shape as the cons data in terms of how the columns are set up.

In [ ]:
test = pd.concat([cons.iloc[0], ind_agg], axis=1).transpose()
test

In [ ]:
# This example will calculate the weights for zone 1.
# The process acts on the dataframe of individual observations.
# We'll make a copy of the individuals to preserve the originals.
ind_copy = ind.copy()
ind_copy

In [ ]:
ind_copy['weight'] = np.ones(5)
ind_copy

In [ ]:
ind_copy.info()

In [ ]:
# Now, we must convert the age variable from the categorical data format  created by pd.cut() to string objects.
# The ipfn library can't handle categorical datatypes for some reason.
ind_copy['age'] = ind_copy['age'].astype(str)
ind_copy.info()

In [ ]:
# Now get the aggregates (marginals) and dimmension for zone 1 for age and sex.
# We need this structure to run the ipf method in python.

age = cons.iloc[0,[0,1]]
sex = cons.iloc[0,[2,3]]

aggregates = [age,sex]
dimensions = [['age'],['sex']]

print(aggregates)
print(dimensions)

In [ ]:
#Now, we can use the library to avoid implementing the algorithm from scratch.
# https://github.com/Dirguis/ipfn
from ipfn import ipfn
ipf = ipfn.ipfn(ind_copy,aggregates,dimensions,weight_col='weight',convergence_rate = 1e-15)
out = ipf.iteration()
out

In [ ]:
# Set up individuals
ind_copy_2 = ind.copy()
ind_copy_2['age'] = ind_copy_2['age'].astype(str)


# First, create some intuitive names for the totals
n_zones = len(cons) # number of zones


# Now, let's do this for each zone; maybe not the most efficient loop :)
for i in range(0,n_zones):
    # Make weights column for zone i
    ind_copy_2['weight_' + str(i)] = np.ones(5)
    
    # Now get the aggregates (marginals) for zone i for age and sex.
    age = cons.iloc[i,[0,1]]
    sex = cons.iloc[i,[2,3]]
    
    # Do iterative proportional fitting
    ipf = ipfn.ipfn(ind_copy_2, [age,sex],[['age'],['sex']],weight_col='weight_'+str(i),convergence_rate = 1e-15)
    ind_copy_2 = ipf.iteration()

ind_copy_2

In [ ]:
# Check that the weights obtained make sense.
# Now create the marginal distribution of individuals in each zone.
ind_agg0 = cons.apply(lambda x: 1.0*ind_agg, axis=0).T[0:3].reset_index(drop=True)

ind_agg3 = (ind_agg0 * np.nan).copy()

for i in range(0,n_zones):
    ind_agg3.iloc[i] = ind_cat.apply(lambda x: x*ind_copy_2['weight_'+str(i)],axis=0).sum(axis=0)

ind_agg3

In [ ]:
# Compare above with constraints - success!
cons

In [ ]:
# The weights generated are fractional, to allcoate individuals to zones 
# we need to convert these to integers. Ideally, with a minimum loss of information.

# We'll start with a function for a method called 'proportional probabilities'.
def int_pp(weights):
    # convert to a vector if required
    xv = np.array(weights).ravel()
    # Sample the individuals
    rsum = round(xv.sum())
    xs = np.random.choice(len(xv),int(rsum),True,xv/xv.sum())
    # return the result
    return np.bincount(xs,minlength=len(xv))



In [ ]:
# Run the function five times for two set of values
np.random.seed(24)
values = [[0.333, 0.667, 3], [1.333, 1.333, 1.333]]

for i, v in enumerate(values, 1):
    print(f"Results for set {i}:")
    for j in range(5):
        result = int_pp(v)
        print(f"Iteration {j+1}: {result}")
    print()

In [ ]:
# Lovelace and Ballas (2013) suggest a truncate, replicate, sample 'TRS' approach to deal with this.
# In effect this means that any individual with weight > 1 is sampled at least once.

def int_trs(weights):
    # convert to a vector if required
    xv = np.array(weights).ravel()
    # truncate - just get the integer part of the weight
    xint = np.floor(xv)
    # Get the decimal bit of the weight
    r = xv - xint
    # Work out the deficit population
    frac_sum = round(r.sum())
    # Sample based upon the deficit bit
    xs = np.random.choice(len(xv),int(frac_sum),True,r/r.sum())
    # Get the result of the deficit part
    topup = np.bincount(xs,minlength=len(xv))
    return xint + topup

In [ ]:
# Test this function
np.random.seed(24) # This seed reproduces the answer.
values = [[0.333, 0.667, 3], [1.333, 1.333, 1.333]]

for i, v in enumerate(values, 1):
    print(f"Results for set {i}:")
    for j in range(5):
        result = int_trs(v)
        print(f"Iteration {j+1}: {result}")
    print()

In [ ]:
# Now use the TRS approach to integerisation to generate some microdata for SimpleWorld
# First, get integer weights for area 1
int_weight1 = int_trs(ind_copy_2['weight_0'])
int_weight1

In [ ]:
# Integerised weights correspond to the number of repetitions of a given individual.
# Firstly, expand the weights
def int_expand_vector(weights):
    return np.repeat(range(0,len(weights)),weights.astype(int))

print (int_weight1)
print (int_expand_vector(int_weight1))

In [ ]:
# expand the indices for zone 1
exp_indices = int_expand_vector(int_weight1)
# Generate the microdata from the individuals table
ind_orig.iloc[exp_indices]

In [ ]:
# Now let's put the integeristation and expansion together for all areas

indivs = []
for i in range(0, n_zones):
    # Integerise and expand
    ints = int_expand_vector(int_trs(ind_copy_2['weight_' + str(i)]))
    # Select the relevant individuals using .loc
    temp = ind_orig.loc[ints].copy()
    # Assign the 'zone' column
    temp['zone'] = i
    indivs.append(temp)

ints_df = pd.concat(indivs)
#Resetting the index of the resulting table.
ints_df.reset_index(drop=True, inplace=True)
ints_df